# ADNI

## INIT

In [1]:
from data_model.DataCleaner import DataCleaner
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

dataCleaner = DataCleaner(support_file_path='ADNI_variables_statistics.xlsx')
client = DatalakeClient()

# Delate files

In [ ]:
search = client.query_files(query={'custom.level' : 'raw','custom.source': 'ADNI'})
print([x['object_name'] for x in search['included_files']])

In [69]:

delete_list = []
for level in ['cleaned_01','cleaned_02']:
    search = client.query_files(query={'custom.level' : level,'custom.source': 'ADNI'})
    elenco = [x['object_name'] for x in search['included_files']]
    delete_list += elenco
    print(elenco)


['cleaned/single_file/ADNIMERGE_25Jul2025_01.csv', 'cleaned/single_file/MMSE_25Jul2025_01.csv', 'cleaned/single_file/PTDEMOG_25Jul2025_01.csv', 'cleaned/single_file/ADSP_PHC_BIOMARKER_25Jul2025_01.csv', 'cleaned/single_file/BLCHANGE_25Jul2025_01.csv', 'cleaned/single_file/DXSUM_25Jul2025_01.csv']


Exception: Query failed: No files match the query criteria

In [70]:

print(delete_list)


['cleaned/single_file/ADNIMERGE_25Jul2025_01.csv', 'cleaned/single_file/MMSE_25Jul2025_01.csv', 'cleaned/single_file/PTDEMOG_25Jul2025_01.csv', 'cleaned/single_file/ADSP_PHC_BIOMARKER_25Jul2025_01.csv', 'cleaned/single_file/BLCHANGE_25Jul2025_01.csv', 'cleaned/single_file/DXSUM_25Jul2025_01.csv']


In [ ]:
'''
for file_name in delete_list:
    results = client.delete_file(
        object_name=file_name
    )
    print(results)
'''

{'metadata_deletion': {'deleted_count': 1, 'success': True}, 'minio_deletion': {'bucket': 'aind', 'object_name': 'cleaned/single_file/ADNIMERGE_25Jul2025_01.csv', 'success': True}, 'success': True}


# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [5]:
search = client.query_files(
    query={'custom.level' : 'raw', 'custom.source' : 'ADNI'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


# Support file managment
operazione per popolare il file support file per i file considerati

In [6]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')
for file_name in zip_files.keys():
    df = zip_files[file_name]
    infoSupportFile = InfoSupportFile(support_file, df, file_name)
    # delate the rows of the support file that are not in the df
    support_file, file_code = infoSupportFile.filter_variables()
    if file_code not in support_file['file_code']:
        continue
    # find the population variable code, and if not in support_file, add it
    pop, support_file = infoSupportFile.find_population_variable()
    # get the variable info and add it to the support_file    
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            infoSupportFile.get_varible_info(key)
# save the updated support file
save_df(df_to_save=support_file, output_path=support_file_path)

file_code 'NEUROPATH' non trovato nel support_file.


In [7]:
create_new_support_file(support_file, support_file_path, new_name='ADNI_variables_cleaned1')

## IF SUPPORT FILE already populated

In [ ]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')
# create and save the new_support_file
create_new_support_file(support_file, support_file_path,  new_name='ADNI_variables_cleaned1')

Open the new_support_file and fill in the new variable codes.

# FILE SPECIFIC DATA CLEANING 1


## ADNI MERGE

In [8]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)


In [9]:
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [10]:
df_new = dataset.copy(deep=True) 


In [11]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

In [12]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
processed_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
processed_df = dataCleaner.drop_if_all_none(processed_df, single_column_required)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['VISCODE'] = processed_df['VISCODE'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [13]:
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')
processed_df = dataCleaner.to_date_format(processed_df, ['EXAMDATE'])
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0],['AGE_bl'],'raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE


In [14]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [15]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [16]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## MMSE

In [17]:
file_code = 'MMSE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'MMSE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [18]:
df_new = dataset.copy(deep=True) 

In [19]:
# Important columns
columns_must_be_verified = ['MMSCORE']

In [20]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'VISDATE')
processed_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
processed_df['VISCODE2'] = processed_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [21]:
filtered_df = processed_df[(processed_df['VISCODE2'] == 'sc') | (processed_df['VISCODE2'] == 'f')]
filtered_df = dataCleaner.handle_f_sc_values(filtered_df, processed_df, 'VISCODE2')
filtered_df = dataCleaner.to_date_format(filtered_df, ['VISDATE'])
final_df = dataCleaner.filter_variables(filtered_df, list(zip_files.keys())[0],prefix='raw')  

In [22]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [23]:
file_code

'MMSE'

In [24]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [25]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## PTDEMOG

In [26]:
file_code = 'PTDEMOG'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [27]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

In [28]:
df_new = dataset.copy(deep=True)
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'VISDATE')
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [29]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [30]:
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOB'], birth_year=row['PTDOBYY']), axis=1)

In [31]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
final_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
final_df = dataCleaner.to_date_format(final_df, ['VISDATE', 'PTDOB'])

In [32]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['AGE'], prefix='raw')

In [33]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [34]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [35]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## ADSP_PHC_BIOMARKER

In [36]:
file_code = 'ADSP_PHC_BIOMARKER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADSP_PHC_BIOMARKER'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [37]:
df_new = dataset.copy(deep=True)

In [38]:
columns_must_be_verified = ['PHC_Tau', 'PHC_pTau', 'PHC_AB42', 'AT_class']
single_column_required = ['PHC_Diagnosis']

In [39]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [40]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [41]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PHC_Sex')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PHC_Education')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PHC_Ethnicity')
processed_df = dataCleaner.convert_to_two_bit(processed_df, col_name='AT_class')
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='PHC_Diagnosis')

In [42]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
final_df['PHC_Race'] = final_df['PHC_Race'].map(mapping)

In [43]:
final_df =dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], prefix='raw')

In [44]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [45]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [46]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## BLCHANGE

In [47]:
file_code = 'BLCHANGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'BLCHANGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [48]:
df_new = dataset.copy(deep=True)

In [49]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [50]:
columns_must_be_verified = ['BCMMSE', 'BCADAS', 'BCPREDX']

In [51]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'EXAMDATE')
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [52]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [53]:
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='BCPREDX')
final_df = dataCleaner.to_date_format(final_df, ['EXAMDATE'])

In [54]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], prefix='raw')

In [55]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [56]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [57]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## DXSUM

In [58]:
file_code = 'DXSUM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
        }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

In [59]:
df_new = dataset.copy(deep=True)

In [60]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [61]:
columns_must_be_verified = ['DXNORM', 'DXMCI', 'DXNODEP']
required_column = ['DIAGNOSIS']

In [62]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'EXAMDATE')
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, required_column)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [63]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [64]:
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DIAGNOSIS')
final_df = dataCleaner.to_date_format(final_df, ['EXAMDATE'])

In [65]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], prefix='raw')

In [66]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [67]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [68]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)